In [ ]:
import numpy as np
import scipy.io as sio
import os
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.stats import wilcoxon
from scipy.stats import mannwhitneyu
import random
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

import torch

In [ ]:
#create pandas dataframe from data

In [ ]:
df_sessions_m1_phase1 = {}

directory = '/home/dvoina/myproj1/pooya_data/M1_Phase2_Lf_gratings/'
files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
files_split = [f.split('_') for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

In [ ]:
new_files = []
for i, f in enumerate(files_split):
    new_files.append([])
    new_files[i].append(int(f[0][1:]))
    new_files[i].append(int(f[1][1:]))
new_files = np.array(new_files)

In [ ]:
n_sessions = len(np.unique(np.array(new_files[:, 1])))

In [ ]:
list_of_stim = ["circ", "circ", "rad", "rad"]*7
list_of_snr = [1.0]*4 + [0.8]*4 + [0.65]*4 + [0.5]*4 + [0.35]*4 + [0.2]*4 + [0.05]*4
list_of_pos = ["R", "L"] * 14

In [ ]:
def get_neuron_rows(n, filename):
    mat_contents = sio.loadmat(directory + filename)

    #print("NEURON N", n)
    total_rows = 0
    list_of_stim_ = [1 if list_of_stim[i] == "circ" else 0 for i in range(len(list_of_stim)) ]
    fr_filter = np.ones(50)
    
    rows = []
    for trial in range(28):

        #print("trial", trial)
        spikes = mat_contents["Spike_mat"][0][trial]["spikes"]
        errors = mat_contents["Spike_mat"][0][trial]["errors"]
        rt = mat_contents["Spike_mat"][0][trial]["saccade_t"]
        
        n_trials = spikes.shape[0]
        #print("n_trials", n_trials)

        for t in range(n_trials):
            
            if errors[t][0] == 0:
                choice = list_of_stim_[trial]
            else:
                choice = 1 - list_of_stim_[trial]
                
            rows.append({
                "neuron_id": n,
                "spike_ts": np.array(spikes[t,:]).flatten(),
                "firing": np.array(spikes[t,:]).flatten()[500:850].sum(), #/(850-500),
                "firing_ts": np.convolve(fr_filter, np.array(spikes[t,:]).flatten()[575:875], "valid"),
                "stim": list_of_stim[trial],
                "snr": list_of_snr[trial],
                "pos": list_of_pos[trial],
                "error": errors[t][0],
                "choice": choice,
                "reaction_time": rt[t][0],
                "trial": total_rows
            })

            total_rows += 1

    #print("total_rows", total_rows)
    return rows

dict_of_neurons = {}
for session in np.unique(new_files[:, 1]):
    ind_s = np.where(new_files[:, 1] == session)[0]
    neurons = new_files[ind_s, 0]
    dict_of_neurons[str(session)] = neurons
    
    #print("neuron number", neurons, len(neurons))
    files_s = [files[i] for i in ind_s]
    #print("session", session, neurons, files_s)

    all_rows = []
    for n, filename in zip(neurons, files_s):
        all_rows.extend(get_neuron_rows(n, filename))
        #print("len all rows", len(all_rows))
        
    df_sessions_m1_phase1[str(session)] = pd.DataFrame(all_rows, columns=["neuron_id", "spike_ts", "firing", "firing_ts", "stim", "snr", "pos", "error", "choice", "reaction_time", "trial"])
    

In [ ]:
def find_tuned_neurons(df, session, snr_list=[1.0, 0.8, 0.65, 0.5, 0.35, 0.2, 0.05]):
    
    neurons = np.unique(df[str(session)].neuron_id.values)
    neurons_tuned_circ = []
    neurons_tuned_rad = []
    neurons_untuned = []
    
    for n in neurons:

        circular_firings = []
        radial_firings = []
        for SNR in snr_list:
            circular = df_sessions_m1_phase1[str(session)][(df[str(session)].stim == 'circ') & (df[str(session)].snr == SNR) & (df[str(session)].neuron_id == n)]
            radial = df_sessions_m1_phase1[str(session)][(df[str(session)].stim == 'rad') & (df[str(session)].snr == SNR) & (df[str(session)].neuron_id == n)]

            #print(len(circular.firing.values), len(radial.firing.values))
            
            circular_firings += list(circular.firing.values)
            radial_firings += list(radial.firing.values)

        stat, p_value1 = mannwhitneyu(circular_firings, radial_firings, alternative='greater')
        stat, p_value2 = mannwhitneyu(radial_firings, circular_firings, alternative='greater')

        if p_value1<=0.05:
            neurons_tuned_circ.append(n)
        if p_value2<=0.05:
            neurons_tuned_rad.append(n)

        if (p_value1 > 0.05) and (p_value2 > 0.05):
            neurons_untuned.append(n)
            
    return neurons_tuned_circ, neurons_tuned_rad, neurons_untuned

neurons_tuned_circ, neurons_tuned_rad, neurons_untuned = find_tuned_neurons(df_sessions_m1_phase1, 3)
tunings = [neurons_tuned_circ, neurons_tuned_rad, neurons_untuned]
neurons_tuned_circ, neurons_tuned_rad, neurons_untuned

In [ ]:
df_all_sessions_m1_phase1 = pd.DataFrame([], columns=["neuron_id", "spike_ts", "firing", "firing_ts", "stim", "snr", "pos", "error", "choice", "reaction_time", "trial"])
session_n = pd.DataFrame([], columns=["session"])

total_neurons = 0
for session in range(1,n_sessions+1):
    df_sessions_m1_phase1[str(session)].loc[:, 'neuron_id'] = df_sessions_m1_phase1[str(session)].neuron_id.copy() + total_neurons
    total_neurons += len(np.unique(np.array(df_sessions_m1_phase1[str(session)].neuron_id)))
    
    df_all_sessions_m1_phase1 = pd.concat([df_all_sessions_m1_phase1, df_sessions_m1_phase1[str(session)]], axis=0)
    session_n = pd.concat([session_n, pd.DataFrame([session]*len(df_sessions_m1_phase1[str(session)].neuron_id.values), columns=["session"])], axis=0)

df_all_sessions_m1_phase1 = pd.concat([df_all_sessions_m1_phase1, session_n], axis=1)

In [ ]:
len(df_all_sessions_m1_phase1["firing_ts"].iloc[0])

In [ ]:
n_neurons = len(df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == "circ") & (df_all_sessions_m1_phase1.snr == 1.0)].neuron_id.unique())
n_trials = 1000
 
list_of_stim = ["circ", "rad"]
list_of_stim = list_of_stim * (n_trials//len(list_of_stim))
snr_choice = 1.0

In [ ]:
X_train = np.zeros((n_trials, n_neurons))
y_train = np.zeros(n_trials)
for trial in range(n_trials):
    print("trial", trial)
    for neuron in range(n_neurons):

        trials = df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == list_of_stim[trial]) & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.neuron_id == neuron+1)].trial
        #print("neuron", neuron+1)
        #print("number of trials", len(trials))

        selected_trial = trial % len(trials)
        #print("sel trial", selected_trial)
        firing = df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == list_of_stim[trial]) & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.neuron_id == neuron+1)].iloc[selected_trial].firing
            
        X_train[trial, neuron] = firing

        if list_of_stim[trial] == "circ":
            y_train[trial] = 1
        elif list_of_stim[trial] == "rad":
            y_train[trial] = 0
        else:
            print("ERROR")
            break

In [ ]:
type(X_train)

In [ ]:
X_centered = X_train - np.mean(X_train, axis=0, keepdims=True)
pca = PCA(n_components=2)
pca.fit(X_centered)


In [ ]:
print(pca.explained_variance_ratio_)
print(pca.singular_values_)
plt.plot(pca.singular_values_, "*-")

In [ ]:
print("pca.components_.shape", pca.components_.shape)
pca.__dict__.keys()


In [ ]:
projections_pca = pca.components_ @ X_centered.T
projections_pca.shape

In [ ]:
plt.plot(projections_pca[0,:], projections_pca[1,:], "*")

In [ ]:
plt.plot(projections_pca[0,:], "*-")
plt.plot(projections_pca[1,:], "*-")

In [ ]:
ind0 = np.where(y_train == 0)[0]; ind1 = np.where(y_train == 1)[0];

plt.plot(projections_pca[0,ind0], projections_pca[1,ind0], "*")
plt.plot(projections_pca[0,ind1], projections_pca[1,ind1], "*")


In [ ]:
plt.plot(pca.components_[0,:])
plt.plot(pca.components_[1,:])


In [ ]:
#for one session compute PCA

In [ ]:
n_neurons = len(df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == "circ") & (df_all_sessions_m1_phase1.snr == 1.0)].neuron_id.unique())
n_trials = 1000
 
list_of_stim = ["circ", "rad"]
list_of_stim = list_of_stim * (n_trials//len(list_of_stim))


In [ ]:
sess = 7
snr_choice = 1.0

list_of_neurons_s = df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == 'circ') & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.session == sess)].neuron_id.unique()
n_trials_s_circ = len(df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == 'circ') & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.session == sess)].trial.unique()) 
n_trials_s_rad = len(df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == 'rad') & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.session == sess)].trial.unique()) 
n_trials_s = n_trials_s_circ + n_trials_s_rad

n_neurons_s = len(list_of_neurons_s)
animal_choice = np.zeros(n_trials_s)

print("list of neurons:", list_of_neurons_s)
print("number of trials: ", n_trials_s, "number of neurons: ", n_neurons_s)

X_train = np.zeros((n_trials_s, n_neurons_s, 251))
y_train = np.zeros(n_trials_s)

trial = 0
for stimulus in ["circ", "rad"]:
    
    print("stimulus", stimulus)
    trials = df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == stimulus) & (df_all_sessions_m1_phase1.snr == snr_choice)  & (df_all_sessions_m1_phase1.session == sess) & (df_all_sessions_m1_phase1.neuron_id == list_of_neurons_s[0])].trial.unique()
    print("len(trials)", len(trials))
    
    for t in range(len(trials)):
        selected_trial =  trial
        for neuron_ind, neuron in enumerate(np.sort(np.array(list_of_neurons_s))):

            #print("neuron", neuron)
            firing = df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == stimulus) & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.session == sess) & (df_all_sessions_m1_phase1.neuron_id == neuron)].iloc[t].firing_ts
            X_train[trial, neuron_ind,:] = firing

            choice = df_all_sessions_m1_phase1[(df_all_sessions_m1_phase1.stim == stimulus) & (df_all_sessions_m1_phase1.snr == snr_choice) & (df_all_sessions_m1_phase1.session == sess) & (df_all_sessions_m1_phase1.neuron_id == neuron)].iloc[t].choice
            animal_choice[trial] = choice
            
            if stimulus == "circ":
                y_train[trial] = 1
            elif stimulus == "rad":
                y_train[trial] = 0
            else:
                print("ERROR")
                break

        trial += 1
                    

In [ ]:
X_train.shape

In [ ]:

X_flat = X_train.transpose(1, 0, 2).reshape(n_neurons_s, -1)  # (n_neurons_s, T*t)

# Subtract neuron-wise mean across trials*time
X_centered = X_flat - X_flat.mean(axis=1, keepdims=True)

# Covariance: neurons x neurons
cov = X_centered @ X_centered.T / (X_flat.shape[1] - 1)

# PCA
n_pcs = 7
eigvals, eigvecs = np.linalg.eigh(cov)

plt.plot(eigvals[::-1], "*-")

# sort descending
idx = np.argsort(eigvals)[::-1]
PCs = eigvecs[:, idx[:n_pcs]]  # (N, n_pcs)

# Project back: trials x n_pcs x time
X_proj = PCs.T @ X_centered  # (n_pcs, T, t)
X_proj = X_proj.reshape(n_pcs, n_trials_s, 251).transpose(1, 0, 2)  # (T, n_pcs, t)

In [ ]:
ind0 = np.where(y_train == 0)[0]
ind1 = np.where(y_train == 1)[0]

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for i0 in range(len(ind0)):
    ax.plot(X_proj[ind0[i0], 0, :], X_proj[ind0[i0], 1, :], X_proj[ind0[i0], 2, :], color='blue')

for i1 in range(len(ind1)):
    ax.plot(X_proj[ind1[i1], 0, :], X_proj[ind1[i1], 1, :], X_proj[ind0[i0], 2, :], color='orange')

In [ ]:
for i0 in range(len(ind0)):
    plt.plot(X_proj[ind0[i0], 0, :], X_proj[ind0[i0], 1, :], color='blue')

for i1 in range(len(ind1)):
    plt.plot(X_proj[ind1[i1], 0, :], X_proj[ind1[i1], 1, :], color='orange')

In [ ]:
# ─────────────────────────────────────────────
# Step 1: Compute the task-variable regressors
# ─────────────────────────────────────────────
# X_train: (n_trials, n_neurons, n_time) — already built
# y_train: (n_trials,)        — 1=circ, 0=rad  (stimulus identity)
# animal_choice: (n_trials,)  — animal's actual choice

# Mante & Sussillo regress each neuron's PSTH (averaged across time)
# against task variables to find the "regression axes"

# Time-average the neural responses per trial → (n_trials, n_neurons)
X_tavg = X_train.mean(axis=2)  # (n_trials, n_neurons)

# Build design matrix: [stimulus, choice]
# You can add more regressors (e.g. correct/incorrect, SNR) if needed
task_vars = np.stack([
    y_train,          # stimulus axis: circ=1, rad=0
    animal_choice,    # choice axis:   animal's response
], axis=1)  # (n_trials, n_task_vars)

# ─────────────────────────────────────────────
# Step 2: Regress each neuron against task vars
# ─────────────────────────────────────────────
# This gives a (n_task_vars, n_neurons) matrix of regression coefficients
# Each row is a vector in neuron space pointing in the direction
# most correlated with that task variable

# Normalize task variables (important for comparing axes)
scaler = StandardScaler()
task_vars_normed = scaler.fit_transform(task_vars)  # (n_trials, n_task_vars)

# Fit regression: predict each neuron's activity from task vars
reg = LinearRegression(fit_intercept=True)
reg.fit(task_vars_normed, X_tavg)  # fits (n_trials, n_neurons)

# Regression coefficient matrix: (n_task_vars, n_neurons)
# Row 0 = stimulus axis in neuron space
# Row 1 = choice axis in neuron space
beta = reg.coef_.T  # LinearRegression gives (n_neurons, n_task_vars), so transpose
# beta shape: (n_task_vars, n_neurons)

beta_stim   = beta[0]  # (n_neurons,) — stimulus regression axis
beta_choice = beta[1]  # (n_neurons,) — choice regression axis

# ─────────────────────────────────────────────
# Step 3: Orthogonalize within the PCA subspace
# ─────────────────────────────────────────────
# Mante & Sussillo project regression axes INTO the PCA subspace first,
# then orthogonalize them (so axes are orthonormal and live in the
# same space as the trajectory projections)

# PCs: (n_neurons, n_pcs) — already computed
# Project regression vectors into PC subspace
beta_stim_pc   = PCs.T @ beta_stim    # (n_pcs,)
beta_choice_pc = PCs.T @ beta_choice  # (n_pcs,)

# Orthogonalize via Gram-Schmidt
def gram_schmidt(vectors):
    """Orthonormalize a list of vectors."""
    basis = []
    for v in vectors:
        w = v.copy()
        for b in basis:
            w -= np.dot(w, b) * b
        norm = np.linalg.norm(w)
        if norm > 1e-10:
            basis.append(w / norm)
    return np.array(basis)  # (n_vectors, n_dims)

# Orthonormal basis in PC space: stimulus axis first, then choice
axes_pc = gram_schmidt([beta_stim_pc, beta_choice_pc])
# axes_pc: (2, n_pcs)

# ─────────────────────────────────────────────
# Step 4: Project trials onto task axes
# ─────────────────────────────────────────────
# X_proj: (n_trials, n_pcs, n_time) — already computed from PCA step

# Project onto stimulus and choice axes through time
proj_stim   = np.tensordot(axes_pc[0], X_proj, axes=[[0],[1]])  # (n_trials, n_time)
if len(axes_pc) > 1:
    proj_choice = np.tensordot(axes_pc[1], X_proj, axes=[[0],[1]])  # (n_trials, n_time)

# ─────────────────────────────────────────────
# Step 5: Plot
# ─────────────────────────────────────────────

time_axis = np.linspace(600, 850, 251)  # ms

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for trial_idx in range(n_trials_s):
    color = 'royalblue' if y_train[trial_idx] == 1 else 'tomato'  # circ vs rad
    ls    = '-' if animal_choice[trial_idx] == 1 else '--'         # choice
    axes[0].plot(time_axis, proj_stim[trial_idx],   color=color, ls=ls, alpha=0.4, lw=0.8)
    if len(axes_pc) > 1:
        axes[1].plot(time_axis, proj_choice[trial_idx], color=color, ls=ls, alpha=0.4, lw=0.8)

# Overlay condition means
for stim_val, label, color in [(1, 'circ', 'royalblue'), (0, 'rad', 'tomato')]:
    mask = y_train == stim_val
    axes[0].plot(time_axis, proj_stim[mask].mean(0),   color=color, lw=2.5, label=label)
    if len(axes_pc) > 1:
        axes[1].plot(time_axis, proj_choice[mask].mean(0), color=color, lw=2.5, label=label)

axes[0].set_title('Projection onto stimulus axis'); axes[0].set_xlabel('Time (ms)')
axes[1].set_title('Projection onto choice axis');   axes[1].set_xlabel('Time (ms)')
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────
# Bonus: 2D state-space plot (stim axis vs choice axis)
# ─────────────────────────────────────────────

if len(axes_pc) > 1:
    fig, ax = plt.subplots(figsize=(6,6))
    for trial_idx in range(n_trials_s):
        color = 'royalblue' if y_train[trial_idx] == 1 else 'tomato'
        ax.plot(proj_stim[trial_idx], proj_choice[trial_idx], color=color, alpha=0.3, lw=0.8)
    
    # Condition means
    for stim_val, label, color in [(1, 'circ', 'royalblue'), (0, 'rad', 'tomato')]:
        mask = y_train == stim_val
        ax.plot(proj_stim[mask].mean(0), proj_choice[mask].mean(0), color=color, lw=2.5, label=label)
    
    ax.set_xlabel('Stimulus axis'); ax.set_ylabel('Choice axis')
    ax.set_title('State space: stimulus vs choice axes')
    ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# X_proj:       (n_trials, n_pcs, n_time)
# y_train:      (n_trials,) — stimulus identity
# animal_choice:(n_trials,) — animal's choice
# errors:       (n_trials,) — 1 if error, 0 if correct
# IMPORTANT: trials must be in chronological order within each session

errors = (y_train != animal_choice).astype(int)  # define if not already

same_stim = (y_train[:-1] == y_train[1:])

# Time-average state per trial
X_tavg = X_proj.mean(axis=2)  # (n_trials, n_pcs)

# ── Build consecutive trial pairs ──────────────────────────────────────
# dX[i] = X(trial i+1) - X(trial i), shape (n_trials-1, n_pcs)
dX_trial = np.diff(X_tavg, axis=0)  # (n_trials-1, n_pcs)
dX_trial = dX_trial[same_stim]

# Error label for each *transition* — was trial t an error?
err_t = errors[:-1]   # error status of the "from" trial
err_t = err_t[same_stim]
# (you could also use errors[1:] — was trial t+1 post-error — depending on hypothesis)

# ── Split transitions by error status ──────────────────────────────────
dX_post_error   = dX_trial[err_t == 1]  # transitions FROM error trials
dX_post_correct = dX_trial[err_t == 0]  # transitions FROM correct trials

print(f"Post-error transitions:   {dX_post_error.shape[0]}")
print(f"Post-correct transitions: {dX_post_correct.shape[0]}")

# ── Mean shift in each condition ────────────────────────────────────────
mean_dX_err  = dX_post_error.mean(axis=0)    # (n_pcs,)
mean_dX_corr = dX_post_correct.mean(axis=0)  # (n_pcs,)

# The "error perturbation axis" — difference of mean shifts
error_perturbation = mean_dX_err - mean_dX_corr  # (n_pcs,)
error_axis = error_perturbation / np.linalg.norm(error_perturbation)

# ── Statistical test: are the two distributions different? ─────────────
from scipy import stats

# Multivariate: per-PC t-tests (Bonferroni corrected)
for pc in range(n_pcs):
    t, p = stats.ttest_ind(dX_post_error[:, pc], dX_post_correct[:, pc])
    print(f"PC{pc+1}: t={t:.3f}, p={p:.4f} (Bonferroni: {min(p*n_pcs,1):.4f})")

# Or a single multivariate Hotelling T^2 test
# (more appropriate since PCs may still be correlated)
from scipy.stats import chi2

def hotelling_t2(X1, X2):
    n1, p = X1.shape
    n2    = X2.shape[0]
    mean_diff = X1.mean(0) - X2.mean(0)
    # Pooled covariance
    S1 = np.cov(X1.T)
    S2 = np.cov(X2.T)
    S_pool = ((n1-1)*S1 + (n2-1)*S2) / (n1+n2-2)
    S_inv  = np.linalg.pinv(S_pool)
    T2 = (n1*n2)/(n1+n2) * mean_diff @ S_inv @ mean_diff
    # Convert to F
    F  = T2 * (n1+n2-p-1) / (p*(n1+n2-2))
    df1, df2 = p, n1+n2-p-1
    p_val = 1 - stats.f.cdf(F, df1, df2)
    return T2, F, p_val

T2, F, p_val = hotelling_t2(dX_post_error, dX_post_correct)
print(f"\nHotelling T²={T2:.3f}, F={F:.3f}, p={p_val:.4f}")

# ── Align error axis with stimulus/choice axes ─────────────────────────
print(f"\nAlignment of error axis with stimulus axis: "
      f"{np.dot(error_axis, axes_pc[0]):.3f}")
print(f"Alignment of error axis with choice axis:  "
      f"{np.dot(error_axis, axes_pc[1]):.3f}")

# ── Visualize ──────────────────────────────────────────────────────────
fig, axes_plot = plt.subplots(1, 2, figsize=(12, 4))

# Project dX onto error axis for each transition
proj_err  = dX_post_error  @ error_axis
proj_corr = dX_post_correct @ error_axis

axes_plot[0].hist(proj_corr, bins=15, alpha=0.6, label='post-correct', color='royalblue')
axes_plot[0].hist(proj_err,  bins=15, alpha=0.6, label='post-error',   color='tomato')
axes_plot[0].axvline(proj_corr.mean(), color='royalblue', lw=2, ls='--')
axes_plot[0].axvline(proj_err.mean(),  color='tomato',    lw=2, ls='--')
axes_plot[0].set_xlabel('Projection onto error axis')
axes_plot[0].set_title('Distribution of trial-to-trial shifts')
axes_plot[0].legend()

# 2D: error axis vs stimulus axis
proj_stim_err  = dX_post_error  @ axes_pc[0]
proj_stim_corr = dX_post_correct @ axes_pc[0]

axes_plot[1].scatter(proj_stim_corr, proj_corr, alpha=0.5, label='post-correct', color='royalblue')
axes_plot[1].scatter(proj_stim_err,  proj_err,  alpha=0.5, label='post-error',   color='tomato')
axes_plot[1].set_xlabel('Stimulus axis'); axes_plot[1].set_ylabel('Error axis')
axes_plot[1].set_title('Trial-to-trial shift: stimulus vs error subspace')
axes_plot[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
X_proj.shape

In [ ]:
same_stim

In [ ]:
#create X_train

snr_choice  = [1,0.8,0.65]#[1,0.8,0.65,0.5,0.35,0.2,0.05] #0.65, 0.5, 0.35, 0.2, 0.05]
sess_choice =  [9,10,11,12] #[7, 8, 9, 10, 11, 12]#[1,2,3,4,5,6,7, 8, 9, 10, 11, 12] #list(range(1,13))

df_filtered = df_all_sessions_m1_phase1[
    df_all_sessions_m1_phase1.snr.isin(snr_choice) &
    df_all_sessions_m1_phase1.session.isin(sess_choice)
]

# Union of all neurons across all sessions
all_neurons   = np.sort(df_filtered.neuron_id.unique())
n_neurons_s   = len(all_neurons)
neuron_to_idx = {n: i for i, n in enumerate(all_neurons)}

# Build trial index from unique (session, stim, trial) combos
trial_index = df_filtered[['session', 'stim', 'snr', 'trial']]\
    .drop_duplicates()\
    .sort_values(['session', 'trial'])\
    .reset_index(drop=True)

n_trials_s = len(trial_index)
print(f"Total trials: {n_trials_s}, total neurons (union): {n_neurons_s}")

X_train        = np.zeros((n_trials_s, n_neurons_s, 251))
y_train        = np.zeros(n_trials_s)
animal_choice  = np.zeros(n_trials_s)
session_labels = np.zeros(n_trials_s)

for t, row in trial_index.iterrows():
    sess     = row['session']
    stim     = row['stim']
    snr      = row['snr']
    trial_id = row['trial']

    session_labels[t] = sess
    y_train[t]        = 1 if stim == 'circ' else 0

    # Get all neurons for this specific (session, stim, trial)
    trial_data = df_filtered[
        (df_filtered.session == sess) &
        (df_filtered.stim    == stim) &
        (df_filtered.snr    == snr) &
        (df_filtered.trial   == trial_id)
    ]

    for _, neuron_row in trial_data.iterrows():
        neuron_idx = neuron_to_idx[neuron_row.neuron_id]
        X_train[t, neuron_idx, :] = neuron_row.firing_ts
        animal_choice[t]          = neuron_row.choice  # same for all neurons in trial

errors = (y_train != animal_choice).astype(int)
print(f"Error rate: {errors.mean():.3f}, N errors: {errors.sum()}")
print(f"Trials per session:\n{trial_index.groupby('session').size()}")

In [ ]:
#PCA on X_train
# ═══════════════════════════════════════════════════════════
# (1) PCA
# ═══════════════════════════════════════════════════════════

n_trials, n_neurons, n_time = X_train.shape
n_pcs = 10

# Reshape to neurons x (trials * time)
X_flat = X_train.transpose(1, 0, 2).reshape(n_neurons, -1)  # (n_neurons, n_trials*n_time)

# Center
X_centered = X_flat - X_flat.mean(axis=1, keepdims=True)

# Covariance and PCA
cov = X_centered @ X_centered.T / (X_flat.shape[1] - 1)
eigvals, eigvecs = np.linalg.eigh(cov)
idx  = np.argsort(eigvals)[::-1]
PCs  = eigvecs[:, idx[:n_pcs]]  # (n_neurons, n_pcs)

# Variance explained
var_explained = eigvals[idx] / eigvals.sum()
print("Variance explained per PC:", np.round(var_explained[:n_pcs], 3))
print("Cumulative variance explained:", np.round(np.cumsum(var_explained[:n_pcs]), 3))

plt.figure(figsize=(6, 3))
plt.plot(var_explained[:20] * 100, '*-')
plt.xlabel('PC'); plt.ylabel('% variance explained')
plt.title('Scree plot'); plt.tight_layout(); plt.show()

# Reshape centered data back to (n_neurons, n_trials, n_time) then project
X_centered_3d = X_centered.reshape(n_neurons, n_trials, n_time).transpose(1, 0, 2)
# (n_trials, n_neurons, n_time)

# Project centered data onto PCs
X_proj = np.tensordot(PCs.T, X_centered_3d, axes=[[1], [1]])  # (n_pcs, n_trials, n_time)
X_proj = X_proj.transpose(1, 0, 2)   



In [ ]:
#create error axis, and later stim and choice axes
# ═══════════════════════════════════════════════════════════
# (2) ERROR AXIS via multivariate regression
# ═══════════════════════════════════════════════════════════

# Time-average state per trial
X_tavg = X_proj.mean(axis=2)  # (n_trials, n_pcs)

# Trial-to-trial velocity, masking session boundaries
valid  = (session_labels[:-1] == session_labels[1:])   # (n_trials-1,)
dX     = np.diff(X_tavg, axis=0)                       # (n_trials-1, n_pcs)

dX_valid      = dX[valid]                              # (n_valid, n_pcs)
state_valid   = X_tavg[:-1][valid]                     # (n_valid, n_pcs) — state at trial t
stim_valid    = y_train[:-1][valid]                    # (n_valid,)
err_valid     = errors[:-1][valid]                     # (n_valid,) — error on trial t
sess_valid    = session_labels[:-1][valid]             # (n_valid,) for same-stim filter

# Same-stimulus filter: only keep transitions where stim doesn't change
same_stim     = (y_train[:-1][valid] == y_train[1:][valid])
dX_same       = dX_valid[same_stim]
state_same    = state_valid[same_stim]
stim_same     = stim_valid[same_stim]
err_same      = err_valid[same_stim]

print(f"\nValid same-stimulus transitions: {same_stim.sum()}")
print(f"  of which post-error:   {err_same.sum()}")
print(f"  of which post-correct: {(err_same==0).sum()}")

# Design matrix: [current state (n_pcs), stimulus (1), error (1)]
scaler = StandardScaler()
design = np.hstack([
    scaler.fit_transform(state_same),   # (n_valid, n_pcs)
    stim_same.reshape(-1, 1),           # (n_valid, 1)
    err_same.reshape(-1, 1)             # (n_valid, 1)  ← what we care about
])  # (n_valid, n_pcs + 2)

reg = LinearRegression(fit_intercept=True)
reg.fit(design, dX_same)  # predicting (n_valid, n_pcs)
# reg.coef_ shape: (n_pcs, n_pcs+2)

# Error axis = last column, normalized
beta_err  = reg.coef_[:, -1]                        # (n_pcs,)
error_axis = beta_err / np.linalg.norm(beta_err)    # (n_pcs,)

print(f"\nError axis (in PC space): {np.round(error_axis, 3)}")
print(f"R² of regression: {reg.score(design, dX_same):.3f}")




# Build design matrix: [stimulus, choice]
# You can add more regressors (e.g. correct/incorrect, SNR) if needed
task_vars = np.stack([
    y_train,          # stimulus axis: circ=1, rad=0
    animal_choice,    # choice axis:   animal's response
], axis=1)  # (n_trials, n_task_vars)

# ─────────────────────────────────────────────
# Step 2: Regress each neuron against task vars
# ─────────────────────────────────────────────
# This gives a (n_task_vars, n_neurons) matrix of regression coefficients
# Each row is a vector in neuron space pointing in the direction
# most correlated with that task variable

# Normalize task variables (important for comparing axes)
scaler = StandardScaler()
task_vars_normed = scaler.fit_transform(task_vars)  # (n_trials, n_task_vars)

# Fit regression: predict each neuron's activity from task vars
reg = LinearRegression(fit_intercept=True)
reg.fit(task_vars_normed, X_tavg)  # fits (n_trials, n_neurons)

# Regression coefficient matrix: (n_task_vars, n_neurons)
# Row 0 = stimulus axis in neuron space
# Row 1 = choice axis in neuron space
beta = reg.coef_.T  # LinearRegression gives (n_neurons, n_task_vars), so transpose
# beta shape: (n_task_vars, n_neurons)

beta_stim_pc   = beta[0]  # (n_neurons,) — stimulus regression axis
beta_choice_pc = beta[1]  # (n_neurons,) — choice regression axis

# ─────────────────────────────────────────────
# Step 3: Orthogonalize within the PCA subspace
# ─────────────────────────────────────────────
# Mante & Sussillo project regression axes INTO the PCA subspace first,
# then orthogonalize them (so axes are orthonormal and live in the
# same space as the trajectory projections)

# Orthogonalize via Gram-Schmidt
def gram_schmidt(vectors):
    """Orthonormalize a list of vectors."""
    basis = []
    for v in vectors:
        w = v.copy()
        for b in basis:
            w -= np.dot(w, b) * b
        norm = np.linalg.norm(w)
        if norm > 1e-10:
            basis.append(w / norm)
    return np.array(basis)  # (n_vectors, n_dims)

# Orthonormal basis in PC space: stimulus axis first, then choice
axes_pc = gram_schmidt([beta_stim_pc, beta_choice_pc])
# axes_pc: (2, n_pcs)


In [ ]:
# ═══════════════════════════════════════════════════════════
# (3) PROJECT ONTO ERROR AXIS + ANALYSIS
# ═══════════════════════════════════════════════════════════

# Project dX onto error axis
proj_err_axis = dX_same @ error_axis                # (n_valid,)

proj_post_error   = proj_err_axis[err_same == 1]
proj_post_correct = proj_err_axis[err_same == 0]

print(f"\nPost-error   mean projection: {proj_post_error.mean():.4f} "
      f"± {proj_post_error.std():.4f}  (n={len(proj_post_error)})")
print(f"Post-correct mean projection: {proj_post_correct.mean():.4f} "
      f"± {proj_post_correct.std():.4f}  (n={len(proj_post_correct)})")

# ── Statistical tests ────────────────────────────────────────────────
t_stat, p_val = stats.ttest_ind(proj_post_error, proj_post_correct)
print(f"\nt-test on error axis projection: t={t_stat:.3f}, p={p_val:.4f}")

# Permutation test (more robust given potential imbalance)
n_perms   = 5000
obs_diff  = proj_post_error.mean() - proj_post_correct.mean()
all_proj  = proj_err_axis.copy()
perm_diffs = np.zeros(n_perms)

for i in range(n_perms):
    perm_labels    = np.random.permutation(err_same)
    perm_diffs[i]  = all_proj[perm_labels==1].mean() - all_proj[perm_labels==0].mean()

p_perm = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))
print(f"Permutation test: observed diff={obs_diff:.4f}, p={p_perm:.4f}")

# Hotelling T² on full dX vectors
def hotelling_t2(X1, X2):
    n1, p  = X1.shape
    n2     = X2.shape[0]
    mean_diff = X1.mean(0) - X2.mean(0)
    S_pool = ((n1-1)*np.cov(X1.T) + (n2-1)*np.cov(X2.T)) / (n1+n2-2)
    T2     = (n1*n2)/(n1+n2) * mean_diff @ np.linalg.pinv(S_pool) @ mean_diff
    F      = T2 * (n1+n2-p-1) / (p*(n1+n2-2))
    p_val  = 1 - stats.f.cdf(F, p, n1+n2-p-1)
    return T2, F, p_val

T2, F, p_hot = hotelling_t2(dX_same[err_same==1], dX_same[err_same==0])
print(f"Hotelling T²={T2:.3f}, F={F:.3f}, p={p_hot:.4f}")

# ── Alignment with stimulus/choice axes ─────────────────────────────
# (assumes axes_pc already computed from TDR section)
print(f"\nAlignment of error axis with stimulus axis: "
      f"{np.dot(error_axis, axes_pc[0]):.3f}")
print(f"Alignment of error axis with choice axis:  "
      f"{np.dot(error_axis, axes_pc[1]):.3f}")

# ── Plots ────────────────────────────────────────────────────────────
fig, axes_plot = plt.subplots(1, 3, figsize=(15, 4))

# 1. Histogram of projections
axes_plot[0].hist(proj_post_correct, bins=20, alpha=0.6,
                  color='royalblue', label=f'post-correct (n={len(proj_post_correct)})')
axes_plot[0].hist(proj_post_error,   bins=20, alpha=0.6,
                  color='tomato',    label=f'post-error (n={len(proj_post_error)})')
axes_plot[0].axvline(proj_post_correct.mean(), color='royalblue', lw=2, ls='--')
axes_plot[0].axvline(proj_post_error.mean(),   color='tomato',    lw=2, ls='--')
axes_plot[0].set_xlabel('Projection onto error axis')
axes_plot[0].set_title(f'Distribution of trial-to-trial shifts\n'
                        f't-test p={p_val:.3f}, perm p={p_perm:.3f}')
axes_plot[0].legend()

# 2. Permutation null distribution
axes_plot[1].hist(perm_diffs, bins=50, color='gray', alpha=0.7, label='null')
axes_plot[1].axvline(obs_diff, color='red', lw=2, label=f'observed={obs_diff:.3f}')
axes_plot[1].set_xlabel('Mean difference (error - correct)')
axes_plot[1].set_title(f'Permutation test (p={p_perm:.3f})')
axes_plot[1].legend()

# 3. 2D scatter: error axis vs stimulus axis projection
proj_stim_axis = dX_same @ axes_pc[0]
axes_plot[2].scatter(proj_stim_axis[err_same==0], proj_err_axis[err_same==0],
                     alpha=0.5, color='royalblue', label='post-correct')
axes_plot[2].scatter(proj_stim_axis[err_same==1], proj_err_axis[err_same==1],
                     alpha=0.8, color='tomato',    label='post-error', s=80, zorder=5)
axes_plot[2].set_xlabel('Projection onto stimulus axis')
axes_plot[2].set_ylabel('Projection onto error axis')
axes_plot[2].set_title('State space: stimulus vs error subspace')
axes_plot[2].legend()

plt.tight_layout()
plt.show()